# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassan-tech-pro/ml-internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

The code uses the documented starter filters (positive 90-day impressions, content age >=90 days, deduplicated by page ID) and describes heavy-tailed counts with median, P90, and maximum. Per-page rates are not averaged into a portfolio CTR; where CTR is tested below, it is recomputed as total clicks / total impressions ×100 with the denominator shown.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

DATA_RELATIVE_PATH = Path("content_refresh_anonymized.csv")
repo_candidates = [Path.cwd(), *Path.cwd().parents]
data_path = next(
    (root / DATA_RELATIVE_PATH for root in repo_candidates if (root / DATA_RELATIVE_PATH).is_file()),
    None,
)
DATA_URL = (
    "https://raw.githubusercontent.com/Hassan-tech-pro/ml-internship/main/"
    "data/raw/content_refresh_anonymized.csv"
)
if data_path is not None:
    raw_data = pd.read_csv(data_path)
    repo_root = data_path.resolve().parents[2]
else:
    raw_data = pd.read_csv(DATA_URL)
    repo_root = Path.cwd()

required_columns = {
    "content_id",
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "search_volume",
    "trend_direction",
    "trend_pct",
}
missing_columns = required_columns.difference(raw_data.columns)
if missing_columns:
    raise ValueError(f"Starter CSV is missing required columns: {sorted(missing_columns)}")

numeric_columns = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "search_volume",
    "trend_pct",
]
for column in numeric_columns:
    raw_data[column] = pd.to_numeric(raw_data[column], errors="coerce")

pages = (
    raw_data.loc[
        raw_data["impressions_90d"].gt(0)
        & raw_data["content_age_days"].ge(90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)
pages["trend_direction"] = pages["trend_direction"].fillna("unknown").astype(str).str.lower()
pages["freshness_bucket"] = np.select(
    [
        pages["days_since_last_update"].le(90),
        pages["days_since_last_update"].le(180),
        pages["days_since_last_update"].isna(),
    ],
    ["0-90 days", "91-180 days", "unknown"],
    default="181+ days",
)
pages["position_bucket"] = np.select(
    [
        pages["avg_position"].isna() | pages["avg_position"].le(0),
        pages["avg_position"].le(3),
        pages["avg_position"].le(10),
        pages["avg_position"].le(20),
        pages["avg_position"].le(50),
    ],
    ["no_data", "top_3", "page_1_4_to_10", "striking_11_to_20", "page_3_to_5"],
    default="deep_50_plus",
)
pages["search_volume_bucket"] = np.select(
    [
        pages["search_volume"].isna(),
        pages["search_volume"].le(0),
        pages["search_volume"].lt(100),
        pages["search_volume"].lt(1000),
    ],
    ["missing", "zero", "1-99", "100-999"],
    default="1000+",
)

print(f"Source: {'local starter CSV' if data_path is not None else 'public starter CSV'}")
print(f"Eligible unique pages after documented filters: {len(pages):,}")
print("Traffic-like values are heavy-tailed; tables below use medians/log medians and weighted CTR.")
distribution_table = pd.DataFrame(
    [
        {
            "field": field,
            "n_nonmissing": int(pages[field].notna().sum()),
            "median": float(pages[field].median()) if pages[field].notna().any() else np.nan,
            "p90": float(pages[field].quantile(0.90)) if pages[field].notna().any() else np.nan,
            "max": float(pages[field].max()) if pages[field].notna().any() else np.nan,
        }
        for field in ["impressions_90d", "clicks_90d", "search_volume", "days_since_last_update"]
    ]
)
display(distribution_table)
print("Identifier fields are used only to deduplicate and will not be displayed.")

Source: public starter CSV
Eligible unique pages after documented filters: 30,000
Traffic-like values are heavy-tailed; tables below use medians/log medians and weighted CTR.


,field,n_nonmissing,median,p90,max
0,impressions_90d,30000,731.0,12136.4,517715.0
1,clicks_90d,30000,1.0,32.0,4178.0
2,search_volume,27532,10.0,110.0,74000.0
3,days_since_last_update,30000,20.0,104.0,373.0


Identifier fields are used only to deduplicate and will not be displayed.


## 2. Signal tests #1 / #2 / #3 (verdict each)

Each test shows bucket counts. **Verdict floor: n >= 50 in both comparison buckets**; otherwise the verdict is `MIXED` (insufficient evidence). The `trend_direction` bucket is a contemporaneous descriptive outcome, not a future target or feature.

1. **Freshness vs. current decline bucket:** compare the down share in `181+ days` since update against `0-90 days`. `CONFIRMED` if stale pages have at least 3 percentage points more `down`; `OPPOSITE` if at least 3 points less; otherwise `MIXED`.
2. **Position vs. click capture:** among pages with at least 100 impressions, compare weighted CTR for `top_3` vs. `deep_50_plus`. `CONFIRMED` if top-3 weighted CTR is higher; `OPPOSITE` if lower; otherwise `MIXED`.
3. **Search-volume bucket vs. observed impressions:** compare median `log1p(impressions_90d)` for `1000+` vs `1-99` search volume. `CONFIRMED` if the high-volume bucket is higher; `OPPOSITE` if lower; otherwise `MIXED`.

These are observational associations in one snapshot. None establish that changing a feature causes the outcome.

In [2]:
MIN_BUCKET_N = 50


def compare_verdict(n_left, n_right, value_left, value_right, minimum_difference=0.0):
    if min(n_left, n_right) < MIN_BUCKET_N:
        return "MIXED"
    difference = value_left - value_right
    if difference > minimum_difference:
        return "CONFIRMED"
    if difference < -minimum_difference:
        return "OPPOSITE"
    return "MIXED"


freshness_summary = (
    pages.assign(is_down=pages["trend_direction"].eq("down"))
    .groupby("freshness_bucket", dropna=False)
    .agg(
        n=("content_id", "size"),
        down_pages=("is_down", "sum"),
        down_share=("is_down", "mean"),
        median_impressions_90d=("impressions_90d", "median"),
    )
    .reindex(["0-90 days", "91-180 days", "181+ days", "unknown"], fill_value=0)
    .rename_axis("freshness_bucket")
    .reset_index()
)
print("Signal 1: refresh freshness and current trend bucket. Values are shares, with page counts n.")
display(freshness_summary)
freshness_index = freshness_summary.set_index("freshness_bucket")
fresh_old = freshness_index.loc["181+ days"]
fresh_recent = freshness_index.loc["0-90 days"]
freshness_verdict = compare_verdict(
    fresh_old["n"], fresh_recent["n"], fresh_old["down_share"], fresh_recent["down_share"], 0.03
)
print(
    f"Freshness verdict: {freshness_verdict} — stale down share "
    f"{fresh_old['down_share']:.1%} (n={int(fresh_old['n']):,}) vs recent "
    f"{fresh_recent['down_share']:.1%} (n={int(fresh_recent['n']):,})."
)

position_pages = pages.loc[
    pages["impressions_90d"].ge(100) & pages["position_bucket"].ne("no_data")
].copy()
position_summary = (
    position_pages.groupby("position_bucket", dropna=False)
    .agg(
        n=("content_id", "size"),
        total_impressions=("impressions_90d", "sum"),
        total_clicks=("clicks_90d", "sum"),
    )
    .reindex(
        ["top_3", "page_1_4_to_10", "striking_11_to_20", "page_3_to_5", "deep_50_plus"],
        fill_value=0,
    )
    .rename_axis("position_bucket")
    .reset_index()
)
position_summary["weighted_ctr_pct"] = np.where(
    position_summary["total_impressions"].gt(0),
    100.0 * position_summary["total_clicks"] / position_summary["total_impressions"],
    np.nan,
)
print("Signal 2: position and click capture, restricted to >=100 impressions per page.")
print("Weighted CTR = total clicks / total impressions ×100; n is page count.")
display(position_summary)
position_index = position_summary.set_index("position_bucket")
pos_top = position_index.loc["top_3"]
pos_deep = position_index.loc["deep_50_plus"]
position_verdict = compare_verdict(
    pos_top["n"], pos_deep["n"], pos_top["weighted_ctr_pct"], pos_deep["weighted_ctr_pct"], 0.05
)
print(
    f"Position verdict: {position_verdict} — top-3 weighted CTR "
    f"{pos_top['weighted_ctr_pct']:.3f}% (n={int(pos_top['n']):,}) vs deep "
    f"{pos_deep['weighted_ctr_pct']:.3f}% (n={int(pos_deep['n']):,})."
)

search_volume_order = ["1-99", "100-999", "1000+", "zero", "missing"]
search_volume_summary = (
    pages.groupby("search_volume_bucket", dropna=False)
    .agg(
        n=("content_id", "size"),
        median_search_volume=("search_volume", "median"),
        median_impressions_90d=("impressions_90d", "median"),
        median_log_impressions=("impressions_90d", lambda values: np.log1p(values).median()),
    )
    .reindex(search_volume_order, fill_value=0)
    .rename_axis("search_volume_bucket")
    .reset_index()
)
print("Signal 3: keyword search-volume buckets vs observed impressions; log1p is used for the heavy tail.")
display(search_volume_summary)
search_index = search_volume_summary.set_index("search_volume_bucket")
sv_high = search_index.loc["1000+"]
sv_low = search_index.loc["1-99"]
volume_verdict = compare_verdict(
    sv_high["n"], sv_low["n"], sv_high["median_log_impressions"], sv_low["median_log_impressions"], 0.10
)
print(
    f"Search-volume verdict: {volume_verdict} — median log impressions "
    f"{sv_high['median_log_impressions']:.3f} (n={int(sv_high['n']):,}) vs "
    f"{sv_low['median_log_impressions']:.3f} (n={int(sv_low['n']):,})."
)

Signal 1: refresh freshness and current trend bucket. Values are shares, with page counts n.


,freshness_bucket,n,down_pages,down_share,median_impressions_90d
0,0-90 days,20655,10576,0.512031,472.0
1,91-180 days,9171,5604,0.611057,1692.0
2,181+ days,174,82,0.471264,15.5
3,unknown,0,0,0.000000,0.0


Freshness verdict: OPPOSITE — stale down share 47.1% (n=174) vs recent 51.2% (n=20,655).
Signal 2: position and click capture, restricted to >=100 impressions per page.
Weighted CTR = total clicks / total impressions ×100; n is page count.


,position_bucket,n,total_impressions,total_clicks,weighted_ctr_pct
0,top_3,555,7554986,36914,0.488604
1,page_1_4_to_10,8660,89279504,311222,0.348593
2,striking_11_to_20,5876,22774408,79383,0.348562
3,page_3_to_5,6037,34997261,53754,0.153595
4,deep_50_plus,878,1213076,479,0.039486


Position verdict: CONFIRMED — top-3 weighted CTR 0.489% (n=555) vs deep 0.039% (n=878).
Signal 3: keyword search-volume buckets vs observed impressions; log1p is used for the heavy tail.


,search_volume_bucket,n,median_search_volume,median_impressions_90d,median_log_impressions
0,1-99,13402,10.0,874.0,6.774224
1,100-999,2385,210.0,801.0,6.687109
2,1000+,664,2400.0,841.5,6.736365
3,zero,11081,0.0,998.0,6.906755
4,missing,2468,NaN,6.0,1.945910


Search-volume verdict: MIXED — median log impressions 6.736 (n=664) vs 6.774 (n=13,402).


## 3. The flag-linked test

**Flag-linked assumption:** refresh-review workflows commonly use staleness together with existing visibility. The released starter CSV does not contain FlyRank's internal product flags, so this tests the underlying signal's coverage—not whether any production flag is correct.

For pages at least 180 days since last update, show the share that also has at least 300 trailing-90-day impressions, with `n` and median impressions. Use a 50-page floor; call it `CONFIRMED` only if at least half of the stale pages meet the visibility floor, `OPPOSITE` if fewer than one quarter do, and `MIXED` otherwise.

In [3]:
STALE_SIGNAL_DAYS = 180
VISIBLE_SIGNAL_IMPRESSIONS = 300
stale_pages = pages.loc[pages["days_since_last_update"].ge(STALE_SIGNAL_DAYS)].copy()
stale_pages["visible_at_rule_floor"] = stale_pages["impressions_90d"].ge(
    VISIBLE_SIGNAL_IMPRESSIONS
)

stale_visibility_table = (
    stale_pages.groupby("freshness_bucket", dropna=False)
    .agg(
        n=("content_id", "size"),
        visible_pages=("visible_at_rule_floor", "sum"),
        share_visible=("visible_at_rule_floor", "mean"),
        median_impressions_90d=("impressions_90d", "median"),
        median_days_since_update=("days_since_last_update", "median"),
    )
    .reset_index()
)
print("Flag-linked signal readout: staleness × visibility, not the absent product flag itself.")
display(stale_visibility_table)

stale_n = len(stale_pages)
stale_visible_share = (
    float(stale_pages["visible_at_rule_floor"].mean()) if stale_n else np.nan
)
if stale_n < MIN_BUCKET_N:
    refresh_signal_verdict = "MIXED"
elif stale_visible_share >= 0.50:
    refresh_signal_verdict = "CONFIRMED"
elif stale_visible_share < 0.25:
    refresh_signal_verdict = "OPPOSITE"
else:
    refresh_signal_verdict = "MIXED"
print(
    f"Stale-visible assumption verdict: {refresh_signal_verdict} — "
    f"{stale_visible_share:.1%} of {stale_n:,} stale pages meet the >=300-impression floor."
)
print("This verdict describes candidate coverage only; no internal flag outcomes are in this dataset.")

Flag-linked signal readout: staleness × visibility, not the absent product flag itself.


,freshness_bucket,n,visible_pages,share_visible,median_impressions_90d,median_days_since_update
0,181+ days,174,22,0.126437,15.5,211.0


Stale-visible assumption verdict: OPPOSITE — 12.6% of 174 stale pages meet the >=300-impression floor.
This verdict describes candidate coverage only; no internal flag outcomes are in this dataset.


## 4. What this means in practice

These are descriptions of one filtered snapshot, not causes or universal rules. A `CONFIRMED` result can support a human review cue, not automatic changes; an `OPPOSITE` result is a reason to weaken or drop that cue; `MIXED` means the buckets or sample do not support a clear decision. Search volume is context, not a literal traffic forecast, and weighted CTR applies only to the stated position/volume slice.

In [4]:
verdict_summary = pd.DataFrame(
    [
        {
            "signal": "Freshness vs current trend bucket",
            "verdict": freshness_verdict,
            "decision_support": (
                "Keep stale pages as a human-review cue only."
                if freshness_verdict == "CONFIRMED"
                else "Do not prioritize on staleness alone; inspect visibility and content context."
                if freshness_verdict == "OPPOSITE"
                else "Treat the freshness comparison as inconclusive for this slice."
            ),
        },
        {
            "signal": "Position vs weighted CTR",
            "verdict": position_verdict,
            "decision_support": (
                "A position-aware snippet review may be worth testing on visible pages."
                if position_verdict == "CONFIRMED"
                else "Do not assume top positions capture clicks better in this slice."
                if position_verdict == "OPPOSITE"
                else "Avoid a position-based CTR recommendation from this comparison alone."
            ),
        },
        {
            "signal": "Search-volume bucket vs impressions",
            "verdict": volume_verdict,
            "decision_support": (
                "Use search volume as context alongside observed impressions, not as a traffic forecast."
                if volume_verdict == "CONFIRMED"
                else "Do not use these buckets to prioritize demand without other evidence."
                if volume_verdict == "OPPOSITE"
                else "Treat search-volume association as inconclusive in this slice."
            ),
        },
        {
            "signal": "Stale-visible coverage behind refresh review",
            "verdict": refresh_signal_verdict,
            "decision_support": (
                "Stale-visible pages exist as a measurable review population; the internal flag itself was not tested."
                if refresh_signal_verdict == "CONFIRMED"
                else "The visibility floor captures few stale pages; reconsider the candidate rule."
                if refresh_signal_verdict == "OPPOSITE"
                else "Do not draw a firm conclusion about stale-page coverage."
            ),
        },
    ]
)
print("Signal audit interpretation (descriptive; one starter snapshot):")
display(verdict_summary)
print("No result here establishes that an edit causes recovery or that Google rewards a particular action.")

Signal audit interpretation (descriptive; one starter snapshot):


,signal,verdict,decision_support
0,Freshness vs current trend bucket,OPPOSITE,Do not prioritize on staleness alone; inspect ...
1,Position vs weighted CTR,CONFIRMED,A position-aware snippet review may be worth t...
2,Search-volume bucket vs impressions,MIXED,Treat search-volume association as inconclusiv...
3,Stale-visible coverage behind refresh review,OPPOSITE,The visibility floor captures few stale pages;...


No result here establishes that an edit causes recovery or that Google rewards a particular action.


## Self-check

Do not mark these complete until you run the notebook yourself and review the displayed outputs:

- [ ] Distributions show counts, medians, upper tails, and heavy-tail handling
- [ ] Three signal tables show `n` and verdicts gated by a 50-page floor
- [ ] CTR uses total clicks / total impressions ×100 and states its volume floor
- [ ] The refresh-flag-linked test states that internal flags are not present in the release
- [ ] Practice summary follows the observed verdicts without causal language
- [ ] No client names, URLs, private queries, or row identifiers appear in displayed output
- [ ] Notebook runs top to bottom; outputs are inspected and explained by me
- [ ] Notebook remains local under `work/notebooks/`; no GitHub changes made here